In [0]:
#Load Tables
from pyspark.sql.functions import *

telemetry = spark.table(
    "workspace.predictive_maintenance.silver_telemetry"
)

maint = spark.table(
    "workspace.predictive_maintenance.silver_maint"
)

In [0]:
#Create Maintenance Event Dataset
maint_analysis = telemetry.join(
    maint.select(
        "machineID",
        col("datetime").alias("maintenance_datetime"),
        "comp"
    ),
    "machineID",
    "inner"
)

display(maint_analysis.limit(10))

machineID,datetime,volt,rotate,pressure,vibration,maintenance_datetime,comp
1,2015-01-03T17:00:00.000Z,164.148094555294,429.822143515459,104.532123632465,43.7886815487723,2015-05-05T06:00:00.000Z,comp3
1,2015-01-06T14:00:00.000Z,159.73519483796,459.258302254684,107.091721396633,45.9445196901824,2015-05-05T06:00:00.000Z,comp3
1,2015-01-11T10:00:00.000Z,171.507835975626,375.227291972199,99.8008928273932,38.5986037515356,2015-05-05T06:00:00.000Z,comp3
1,2015-01-13T18:00:00.000Z,151.597773638485,414.104048571942,120.94688310289,35.9637807218198,2015-05-05T06:00:00.000Z,comp3
1,2015-01-17T06:00:00.000Z,193.502481424582,438.385858854957,101.117883146434,34.3379626375048,2015-05-05T06:00:00.000Z,comp3
1,2015-01-19T07:00:00.000Z,194.530211868976,463.046798582402,104.171863686517,39.9099431229969,2015-05-05T06:00:00.000Z,comp3
1,2015-01-21T06:00:00.000Z,155.568214575444,447.273159503477,98.2725794689957,39.9891315557569,2015-05-05T06:00:00.000Z,comp3
1,2015-01-23T08:00:00.000Z,191.903986193189,442.679313561805,96.5276694008562,45.1136575679724,2015-05-05T06:00:00.000Z,comp3
1,2015-01-23T10:00:00.000Z,145.114385444797,453.286889481496,122.005344050685,39.600431484911,2015-05-05T06:00:00.000Z,comp3
1,2015-01-23T20:00:00.000Z,171.793824372364,447.78950671517,92.1802933814706,44.9941161190469,2015-05-05T06:00:00.000Z,comp3


In [0]:
#Label Before and After Maintenance
maint_analysis = maint_analysis.withColumn(
    "maintenance_period",
    when(
        col("datetime") < col("maintenance_datetime"),
        "Before Maintenance"
    ).otherwise(
        "After Maintenance"
    )
)

display(
    maint_analysis.select(
        "machineID",
        "datetime",
        "maintenance_datetime",
        "maintenance_period"
    ).limit(10)
)

machineID,datetime,maintenance_datetime,maintenance_period
1,2015-01-03T17:00:00.000Z,2015-05-05T06:00:00.000Z,Before Maintenance
1,2015-01-06T14:00:00.000Z,2015-05-05T06:00:00.000Z,Before Maintenance
1,2015-01-11T10:00:00.000Z,2015-05-05T06:00:00.000Z,Before Maintenance
1,2015-01-13T18:00:00.000Z,2015-05-05T06:00:00.000Z,Before Maintenance
1,2015-01-17T06:00:00.000Z,2015-05-05T06:00:00.000Z,Before Maintenance
1,2015-01-19T07:00:00.000Z,2015-05-05T06:00:00.000Z,Before Maintenance
1,2015-01-21T06:00:00.000Z,2015-05-05T06:00:00.000Z,Before Maintenance
1,2015-01-23T08:00:00.000Z,2015-05-05T06:00:00.000Z,Before Maintenance
1,2015-01-23T10:00:00.000Z,2015-05-05T06:00:00.000Z,Before Maintenance
1,2015-01-23T20:00:00.000Z,2015-05-05T06:00:00.000Z,Before Maintenance


In [0]:
#Compare Telemetry Metrics
maintenance_impact = maint_analysis.groupBy(
    "maintenance_period"
).agg(
    avg("volt").alias("avg_volt"),
    avg("rotate").alias("avg_rotate"),
    avg("pressure").alias("avg_pressure"),
    avg("vibration").alias("avg_vibration")
)

display(maintenance_impact)

maintenance_period,avg_volt,avg_rotate,avg_pressure,avg_vibration
Before Maintenance,170.8099425155955,446.59896698623265,100.83625919702125,40.38958017790705
After Maintenance,170.75465981988077,446.6056308529128,100.87572407526322,40.38207756144092


Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.